In [ ]:

import json
import math
import os
import subprocess
import gc

import polars as pl
from phetk.phecode import Phecode


pl.Config.set_engine_affinity("streaming")
pl.Config.set_streaming_chunk_size(250_000)

bucket_name = os.getenv("WORKSPACE_BUCKET", "").replace("gs://", "").strip("/")
path_base   = f"gs://{bucket_name}/sjl"
path_out    = f"{path_base}/outputs"
path_ck     = f"{path_base}/checkpoints"

RESET_CHECKPOINTS = True

LANDMARK_DAYS       = 182
MIN_NIGHTS_REQUIRED = 14
MIN_WEEKEND_NIGHTS  = 4
MIN_WEEKDAYS        = 3
MIN_WEEKENDS        = 1
MIN_STEPS_THRESHOLD = 500

MIN_PER_DAY = 1440.0
TWO_PI      = 2 * math.pi


def gcs_exists(path: str) -> bool:
    return subprocess.call(["gsutil", "-q", "stat", path]) == 0


def gcs_write_json(obj: dict, path: str) -> None:
    tmp = "/tmp/_tmp_json.json"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    subprocess.call(["gsutil", "cp", tmp, path])


def inner_join(left, right, on="person_id"):
    l = left.lazy()  if isinstance(left,  pl.DataFrame) else left
    r = right.lazy() if isinstance(right, pl.DataFrame) else right
    return l.join(r, on=on, how="inner")


def lazy_to_dummies(lf: pl.LazyFrame, columns: list) -> pl.LazyFrame:
    for col in columns:
        unique_vals = (
            lf.select(pl.col(col)).unique().collect()
            .get_column(col).to_list()
        )
        unique_vals = [v for v in unique_vals if v is not None]
        exprs = [(pl.col(col) == v).cast(pl.Int8).alias(f"{col}_{v}") for v in unique_vals]
        lf = lf.with_columns(exprs).drop(col)
    return lf


def add_sjl_tertiles(lf: pl.LazyFrame):
    df = lf.select("avg_sjl_hours").collect()
    q1 = float(df["avg_sjl_hours"].quantile(0.333))
    q2 = float(df["avg_sjl_hours"].quantile(0.667))
    lf = lf.with_columns(
        pl.when(pl.col("avg_sjl_hours") <= q1).then(1)
        .when(pl.col("avg_sjl_hours") <= q2).then(2)
        .otherwise(3)
        .cast(pl.Int8).alias("sjl_tertile")
    )
    return lf, q1, q2


def circular_mean_expr(col_name: str, out_name: str) -> pl.Expr:
    """
    Native Polars circular mean for a column of minute-of-day values (0-1439).
    Uses the standard circular mean formula:
        mean_angle = atan2(mean(sin(theta)), mean(cos(theta)))
    where theta = (minutes / 1440) * 2pi
    """
    theta = pl.col(col_name) * (TWO_PI / MIN_PER_DAY)
    return (
        (
            (pl.arctan2(theta.sin().mean(), theta.cos().mean()) + TWO_PI) % TWO_PI
        ) * MIN_PER_DAY / TWO_PI
    ).alias(out_name)


survey_answer_map = {
    "What Race Ethnicity: White":                         "race_white",
    "What Race Ethnicity: Black":                         "race_black",
    "What Race Ethnicity: Asian":                         "race_asian",
    "What Race Ethnicity: Hispanic":                      "race_hispanic",
    "What Race Ethnicity: AIAN":                          "race_other",
    "What Race Ethnicity: NHPI":                          "race_other",
    "What Race Ethnicity: MENA":                          "race_other",
    "What Race Ethnicity: Race Ethnicity Other":          "race_other",
    "What Race Ethnicity: Race Ethnicity None Of These":  "race_other",
    "Highest Grade: Never Attended":       "edu_lt_hs",
    "Highest Grade: One Through Four":     "edu_lt_hs",
    "Highest Grade: Five Through Eight":   "edu_lt_hs",
    "Highest Grade: Nine Through Eleven":  "edu_lt_hs",
    "Highest Grade: Twelve Or GED":        "edu_hs",
    "Highest Grade: College One to Three": "edu_college_plus",
    "Highest Grade: College Graduate":     "edu_college_plus",
    "Highest Grade: Advanced Degree":      "edu_college_plus",
    "Employment Status: Employed For Wages":       "emp_structured",
    "Employment Status: Employed Full Time":       "emp_structured",
    "Employment Status: Self Employed":            "emp_structured",
    "Employment Status: Student":                  "emp_structured",
    "Employment Status: Retired":                  "emp_unstructured",
    "Employment Status: Homemaker":                "emp_unstructured",
    "Employment Status: Out Of Work":              "emp_unstructured",
    "Employment Status: Out Of Work One Or More":  "emp_unstructured",
    "Employment Status: Out Of Work Less Than One":"emp_unstructured",
    "Employment Status: Unable To Work":           "emp_unstructured",
    "100 Cigs Lifetime: Yes": "smoke_ever",
    "100 Cigs Lifetime: No":  "smoke_never",
    "Alcohol Participant: Yes":                     "alcohol_yes",
    "Alcohol Participant: No":                      "alcohol_no",
    "Drink Frequency Past Year: Never":             "alcohol_never",
    "Drink Frequency Past Year: Monthly Or Less":   "alcohol_occasional",
    "Drink Frequency Past Year: 2 to 4 Per Month":  "alcohol_occasional",
    "Drink Frequency Past Year: 2 to 3 Per Week":   "alcohol_regular",
    "Drink Frequency Past Year: 4 or More Per Week":"alcohol_regular",
    "Annual Income: less 10k":   "income_low",
    "Annual Income: 10k 25k":    "income_low",
    "Annual Income: 25k 35k":    "income_mid",
    "Annual Income: 35k 50k":    "income_mid",
    "Annual Income: 50k 75k":    "income_mid",
    "Annual Income: 75k 100k":   "income_mid",
    "Annual Income: 100k 150k":  "income_high",
    "Annual Income: 150k 200k":  "income_high",
    "Annual Income: more 200k":  "income_high",
    "PMI: Skip":                 "unknown",
    "PMI: Prefer Not To Answer": "unknown",
    "PMI: Dont Know":            "unknown",
    "PMI: Don't Know":           "unknown",
}

table1_detail_map = {
    "Highest Grade: Never Attended":       "Less than HS",
    "Highest Grade: One Through Four":     "Less than HS",
    "Highest Grade: Five Through Eight":   "Less than HS",
    "Highest Grade: Nine Through Eleven":  "Less than HS",
    "Highest Grade: Twelve Or GED":        "HS Graduate/GED",
    "Highest Grade: College One to Three": "Some College",
    "Highest Grade: College Graduate":     "College Graduate",
    "Highest Grade: Advanced Degree":      "Advanced Degree",
    "Annual Income: less 10k":   "< $10k",
    "Annual Income: 10k 25k":    "$10k - $25k",
    "Annual Income: 25k 35k":    "$25k - $35k",
    "Annual Income: 35k 50k":    "$35k - $50k",
    "Annual Income: 50k 75k":    "$50k - $75k",
    "Annual Income: 75k 100k":   "$75k - $100k",
    "Annual Income: 100k 150k":  "$100k - $150k",
    "Annual Income: 150k 200k":  "$150k - $200k",
    "Annual Income: more 200k":  "> $200k",
    "Employment Status: Employed For Wages":       "Employed (Wages)",
    "Employment Status: Employed Full Time":       "Employed (Full Time)",
    "Employment Status: Self Employed":            "Self-Employed",
    "Employment Status: Student":                  "Student",
    "Employment Status: Retired":                  "Retired",
    "Employment Status: Homemaker":                "Homemaker",
    "Employment Status: Out Of Work":              "Unemployed",
    "Employment Status: Out Of Work One Or More":  "Unemployed (>1 yr)",
    "Employment Status: Out Of Work Less Than One":"Unemployed (<1 yr)",
    "Employment Status: Unable To Work":           "Unable to Work",
    "PMI: Skip":                 "Unknown",
    "PMI: Prefer Not To Answer": "Unknown",
    "PMI: Dont Know":            "Unknown",
    "PMI: Don't Know":           "Unknown",
    "Skip":                      "Unknown",
}


if RESET_CHECKPOINTS:
    print("Step 0: Resetting checkpoints...")
    subprocess.call(["gsutil", "-m", "rm", "-rf", f"{path_ck}/*"])

lf_person      = pl.scan_parquet(f"{path_base}/data/person_*.parquet")
lf_survey      = pl.scan_parquet(f"{path_base}/data/survey_*.parquet")
lf_sleep       = pl.scan_parquet(f"{path_base}/data/sleepmin_*.parquet")
lf_steps       = pl.scan_parquet(f"{path_base}/data/steps_*.parquet")
lf_ehr_length  = pl.scan_parquet(f"{path_base}/data/ehr_length_*.parquet")
lf_deprivation = pl.scan_parquet(f"{path_base}/data/deprivation_*.parquet")
lf_bmi         = pl.scan_parquet(f"{path_base}/data/bmi_*.parquet")


phecode_file = f"{path_base}/invariant/phecode_counts.tsv"
if not gcs_exists(phecode_file):
    print("Step 1: Computing phecode counts...")
    Phecode(platform="aou").count_phecode(
        phecode_version="X",
        icd_version="US",
        output_file_path=phecode_file,
    )
else:
    print("Step 1: Phecode counts already exist, skipping.")


SURVEY_WIDE = f"{path_ck}/survey_wide.parquet"

if not gcs_exists(SURVEY_WIDE):
    print("Step 2: Building survey wide...")
    detail_targets = [
        "Income: Annual Income",
        "Education Level: Highest Grade",
        "Employment: Employment Status",
    ]
    lf_main   = lf_survey.with_columns(
        pl.col("answer").replace(survey_answer_map).fill_null("unknown")
    )
    lf_detail = (
        lf_survey
        .filter(pl.col("question").is_in(detail_targets))
        .with_columns([
            pl.col("answer").replace(table1_detail_map).fill_null("Unknown"),
            (pl.col("question") + "_detail").alias("question"),
        ])
    )
    lf_combined = pl.concat([lf_main, lf_detail])
    q_levels = (
        lf_combined.select(pl.col("question").cast(pl.Categorical))
        .unique().collect().get_column("question").to_list()
    )
    (
        lf_combined
        .with_columns([
            pl.col("answer").cast(pl.Categorical),
            pl.col("question").cast(pl.Categorical),
            pl.col("survey_datetime").min().over("person_id").alias("enrollment_date"),
        ])
        .select(["person_id", "enrollment_date", "question", "answer", "survey_datetime"])
        .sort(["person_id", "survey_datetime"])
        .pivot(
            on="question", on_columns=q_levels,
            index=["person_id", "enrollment_date"],
            values="answer", aggregate_function="first", maintain_order=True,
        )
        .with_columns(
            Alcohol=pl.when(pl.col("Alcohol: Alcohol Participant") == "alcohol_no")
            .then(pl.lit("alcohol_never"))
            .otherwise(pl.col("Alcohol: Drink Frequency Past Year"))
            .cast(pl.Categorical)
        )
        .drop(["Alcohol: Alcohol Participant", "Alcohol: Drink Frequency Past Year"])
        .with_columns(pl.all().exclude(["person_id", "enrollment_date"]).fill_null("unknown"))
        .sink_parquet(SURVEY_WIDE)
    )
    gc.collect()
else:
    print("Step 2: Survey wide checkpoint exists, skipping.")

wide = pl.scan_parquet(SURVEY_WIDE)


print("Step 3: Building person base...")
consort = {}
consort["n_total"] = lf_person.select(pl.len()).collect().item()

lf_person_sex = (
    lf_person
    .with_columns(
        sex_at_birth=pl.col("sex_at_birth").cast(pl.Utf8).str.strip_chars()
        .replace({"Male": "male", "Female": "female"})
    )
    .filter(pl.col("sex_at_birth").is_in(["male", "female"]))
    .with_columns(
        sex=pl.when(pl.col("sex_at_birth") == "male").then(1).otherwise(0).cast(pl.Int8)
    )
)
consort["n_sex"] = lf_person_sex.select(pl.len()).collect().item()

median_dep = lf_deprivation.select(pl.col("deprivation_index").median()).collect().item()
lf_person_dep = (
    lf_person_sex
    .join(lf_deprivation, on="person_id", how="left")
    .with_columns(pl.col("deprivation_index").fill_null(median_dep))
)

lf_first_sleep = (
    lf_sleep.filter(pl.col("is_main_sleep") == "true")
    .group_by("person_id")
    .agg(pl.col("sleep_date").min().alias("first_sleep_date"))
)

lf_person_sleep = (
    inner_join(lf_first_sleep, lf_person_dep)
    .with_columns(
        age_at_first_fitbit_years=(
            pl.col("first_sleep_date").dt.year() - pl.col("year_of_birth")
        ).cast(pl.Int16)
    )
    .with_columns(age_at_landmark=(pl.col("age_at_first_fitbit_years") + 0.5))
)
consort["n_sleep"] = lf_person_sleep.select(pl.len()).collect().item()

print("Step 4: Daily sleep aggregation...")

lf_sleep_daily = (
    lf_sleep
    .filter(pl.col("is_main_sleep") == "true")
    .with_columns(
        (pl.col("start_datetime") + pl.duration(minutes=pl.col("duration_in_min")))
        .alias("end_datetime")
    )
    .group_by(["person_id", "sleep_date"])
    .agg([
        pl.col("start_datetime").min().alias("sleep_start"),
        pl.col("end_datetime").max().alias("wake_time"),
        pl.col("duration_in_min").sum().alias("total_sleep_min"),
    ])
    .with_columns([
        (pl.col("total_sleep_min") / 60).alias("total_sleep_hours"),
        pl.when(pl.col("sleep_date").dt.weekday() >= 6)
        .then(pl.lit("weekend")).otherwise(pl.lit("weekday"))
        .alias("day_type"),
    ])
)


print("Step 5: Computing landmark windows...")

lf_win_lazy = (
    lf_sleep_daily
    .select(["person_id", "sleep_date", "day_type"])
    .with_columns(
        pl.col("sleep_date").cast(pl.Date).min().over("person_id")
        .alias("landmark_start_date")
    )
    .with_columns(
        (pl.col("landmark_start_date") + pl.duration(days=LANDMARK_DAYS))
        .alias("landmark_end_date")
    )
    .filter(pl.col("sleep_date").is_between(
        pl.col("landmark_start_date"), pl.col("landmark_end_date"), closed="both"
    ))
    .group_by("person_id")
    .agg([
        pl.first("landmark_start_date"),
        pl.first("landmark_end_date"),
        pl.len().alias("n_sleep_nights_in_window"),
        (pl.col("day_type").str.to_lowercase() == "weekend").sum()
        .alias("n_weekend_nights_in_window"),
    ])
    .filter(
        (pl.col("n_sleep_nights_in_window") >= MIN_NIGHTS_REQUIRED) &
        (pl.col("n_weekend_nights_in_window") >= MIN_WEEKEND_NIGHTS)
    )
    .select(["person_id", "landmark_start_date", "landmark_end_date"])
)

df_win = lf_win_lazy.collect()
lf_win = df_win.lazy()


def compute_sjl_from_sleep(
    lf_sleep_daily: pl.LazyFrame,
    lf_window: pl.LazyFrame | None,
) -> tuple[pl.LazyFrame, pl.LazyFrame]:
    """
    Compute SJLsc (sleep-corrected social jetlag) from daily sleep records.

    Parameters
    ----------
    lf_sleep_daily : LazyFrame
        Daily sleep records with sleep_start, wake_time, total_sleep_min, day_type.
    lf_window : LazyFrame or None
        Per-person date window with landmark_start_date and landmark_end_date.
        Pass None to use all available data (all-time mode).

    Returns
    -------
    person_summary : LazyFrame
        One row per person: avg SJL, circular midpoints, duration stats.
    weekly_long : LazyFrame
        One row per (person, week): weekly SJL and midpoint values.
    """

    if lf_window is not None:
        lf_lm = (
            lf_sleep_daily
            .join(lf_window, on="person_id", how="inner")
            .filter(pl.col("sleep_date").is_between(
                pl.col("landmark_start_date"), pl.col("landmark_end_date"), closed="both"
            ))
            .with_columns(pl.col("day_type").str.to_lowercase())
        )
    else:
        lf_lm = lf_sleep_daily.with_columns(pl.col("day_type").str.to_lowercase())

    lf_daily = (
        lf_lm
        .with_columns([
            (pl.col("sleep_date") - pl.duration(
                days=pl.col("sleep_date").dt.weekday() - 1
            )).alias("week_start"),
            (pl.col("sleep_start") + (pl.col("wake_time") - pl.col("sleep_start")) / 2)
            .alias("midpoint_dt"),
        ])
        .with_columns(
            (
                pl.col("midpoint_dt").dt.hour().cast(pl.Int64) * 60
                + pl.col("midpoint_dt").dt.minute()
                + pl.col("midpoint_dt").dt.second() / 60
            ).alias("midpoint_min")
        )
        .with_columns(
            (pl.col("midpoint_min") * (TWO_PI / MIN_PER_DAY)).alias("theta")
        )
    )

    weekly_stats = (
        lf_daily
        .group_by(["person_id", "week_start", "day_type"])
        .agg([
            pl.len().alias("n_days"),
            pl.col("theta").sin().mean().alias("mean_sin"),
            pl.col("theta").cos().mean().alias("mean_cos"),
            pl.col("total_sleep_min").mean().alias("mean_duration_min"),
        ])
        .with_columns(
            (((pl.arctan2("mean_sin", "mean_cos") + TWO_PI) % TWO_PI) * MIN_PER_DAY / TWO_PI)
            .alias("mean_midpoint_min")
        )
    )

    weekly_wide = (
        weekly_stats
        .group_by(["person_id", "week_start"])
        .agg([
            pl.col("mean_midpoint_min").filter(pl.col("day_type") == "weekday").first().alias("msw_min"),
            pl.col("mean_midpoint_min").filter(pl.col("day_type") == "weekend").first().alias("msf_min"),
            pl.col("mean_duration_min").filter(pl.col("day_type") == "weekday").first().alias("dur_weekday_min"),
            pl.col("mean_duration_min").filter(pl.col("day_type") == "weekend").first().alias("dur_weekend_min"),
            pl.col("n_days").filter(pl.col("day_type") == "weekday").first().alias("n_days_weekday"),
            pl.col("n_days").filter(pl.col("day_type") == "weekend").first().alias("n_days_weekend"),
        ])
        .filter(
            (pl.col("n_days_weekday") >= MIN_WEEKDAYS) &
            (pl.col("n_days_weekend") >= MIN_WEEKENDS)
        )
        .with_columns(
            ((pl.col("dur_weekday_min") * 5 + pl.col("dur_weekend_min") * 2) / 7)
            .alias("avg_weekly_dur_min")
        )
        .with_columns(
            (pl.col("msf_min") - ((pl.col("dur_weekend_min") - pl.col("avg_weekly_dur_min")) / 2))
            .alias("msf_sc_min")
        )
        .with_columns(
            (((pl.col("msf_sc_min") - pl.col("msw_min") + 720) % 1440) - 720)
            .alias("sjl_sc_min_signed")
        )
        .with_columns(
            (pl.col("sjl_sc_min_signed").abs() / 60).alias("sjl_sc_hours")
        )
    )

    weekly_long = weekly_wide.select([
        "person_id", "week_start",
        "sjl_sc_hours", "sjl_sc_min_signed",
        "msw_min", "msf_min", "msf_sc_min",
        "dur_weekday_min", "dur_weekend_min",
        "n_days_weekday", "n_days_weekend",
    ])

    avg_sleep = (
        lf_lm.group_by("person_id")
        .agg((pl.col("total_sleep_min").mean() / 60).alias("avg_sleep_duration_hours"))
    )

    person_summary = (
        weekly_wide
        .group_by("person_id")
        .agg([
            circular_mean_expr("msw_min", "mean_msw_min"),
            circular_mean_expr("msf_min", "mean_msf_min"),
            pl.col("sjl_sc_hours").mean().alias("avg_sjl_hours"),
            pl.col("sjl_sc_hours").std().alias("sd_sjl_hours"),
            pl.col("sjl_sc_hours").count().alias("n_weeks"),
            pl.col("dur_weekday_min").mean().alias("mean_dur_weekday_min"),
            pl.col("dur_weekend_min").mean().alias("mean_dur_weekend_min"),
        ])
        .join(avg_sleep, on="person_id", how="inner")
    )

    return person_summary, weekly_long



print("Step 6: Computing landmark SJL...")

lm_person_summary, lm_weekly_long = compute_sjl_from_sleep(
    lf_sleep_daily=lf_sleep_daily,
    lf_window=lf_win,
)

sjl_person = lm_person_summary.join(lf_win, on="person_id", how="inner")


print("Step 7: Computing all-time SJL...")

lf_sleep_daily_cohort = (
    lf_sleep_daily
    .join(df_win.select("person_id").lazy(), on="person_id", how="inner")
)

alltime_person_summary, alltime_weekly_long = compute_sjl_from_sleep(
    lf_sleep_daily=lf_sleep_daily_cohort,
    lf_window=None,  
)

alltime_person_summary = alltime_person_summary.rename({
    "avg_sjl_hours":             "alltime_avg_sjl_hours",
    "sd_sjl_hours":              "alltime_sd_sjl_hours",
    "n_weeks":                   "alltime_n_weeks",
    "mean_msw_min":              "alltime_mean_msw_min",
    "mean_msf_min":              "alltime_mean_msf_min",
    "mean_dur_weekday_min":      "alltime_mean_dur_weekday_min",
    "mean_dur_weekend_min":      "alltime_mean_dur_weekend_min",
    "avg_sleep_duration_hours":  "alltime_avg_sleep_duration_hours",
})


print("Step 8: Computing steps...")

if df_win.height > 0:
    g_min = df_win["landmark_start_date"].min()
    g_max = df_win["landmark_end_date"].max()
    steps_person = (
        lf_steps
        .filter(pl.col("date").is_between(g_min, g_max))
        .join(lf_win, on="person_id", how="inner")
        .filter(
            (pl.col("steps") > MIN_STEPS_THRESHOLD) &
            pl.col("date").is_between(
                pl.col("landmark_start_date"), pl.col("landmark_end_date"), closed="both"
            )
        )
        .group_by("person_id")
        .agg(pl.col("steps").median().alias("median_daily_steps"))
    )
else:
    steps_person = pl.DataFrame({
        "person_id":          pl.Series([], dtype=pl.Int64),
        "median_daily_steps": pl.Series([], dtype=pl.Float64),
    }).lazy()


print("Step 9: Assembling cohort...")

person_survey     = inner_join(wide, lf_person_sleep)
person_survey_ehr = inner_join(person_survey, lf_ehr_length)
consort["n_survey_ehr"] = person_survey_ehr.select(pl.len()).collect().item()

cohort = inner_join(person_survey_ehr, sjl_person)
consort["n_sjl"] = cohort.select(pl.len()).collect().item()

cohort = inner_join(cohort, steps_person)
consort["n_steps"] = cohort.select(pl.len()).collect().item()

cohort = (
    cohort
    .with_columns([
        pl.col("last_observation_date").cast(pl.Date),
        pl.col("landmark_end_date").cast(pl.Date),
    ])
    .with_columns([
        (pl.col("last_observation_date") - pl.col("landmark_end_date"))
        .dt.total_days().alias("followup_days"),
        ((pl.col("last_observation_date") - pl.col("landmark_end_date"))
         .dt.total_days() / 365.25).alias("censoring_time_years"),
    ])
    .filter(pl.col("ehr_length_years") <= 40)
)
consort["n_final"] = cohort.select(pl.len()).collect().item()

consort["ex_sex"]        = consort["n_total"]      - consort["n_sex"]
consort["ex_sleep"]      = consort["n_sex"]        - consort["n_sleep"]
consort["ex_survey_ehr"] = consort["n_sleep"]      - consort["n_survey_ehr"]
consort["ex_sjl"]        = consort["n_survey_ehr"] - consort["n_sjl"]
consort["ex_steps"]      = consort["n_sjl"]        - consort["n_steps"]
consort["ex_ehr_len"]    = consort["n_steps"]      - consort["n_final"]
print("CONSORT:", {k: f"{v:,}" if isinstance(v, int) else v for k, v in consort.items()})


lf_apnea = (
    pl.scan_csv(phecode_file, separator="\t", try_parse_dates=True)
    .filter(pl.col("phecode").str.starts_with("NS_333.1"))
    .select(["person_id", "first_event_date"])
)

cohort = (
    cohort
    .with_columns(pl.col("landmark_start_date").cast(pl.Date))
    .join(lf_apnea, on="person_id", how="left")
    .with_columns(
        (pl.col("first_event_date") < pl.col("landmark_start_date"))
        .fill_null(False).cast(pl.Int8).alias("apnea_flag")
    )
    .group_by(pl.all().exclude("apnea_flag", "first_event_date"))
    .agg(pl.max("apnea_flag").fill_null(0).alias("apnea_flag"))
)


cohort_keys = cohort.select(["person_id", "landmark_start_date"]).collect()

best_bmi = (
    lf_bmi
    .select(["person_id", "measurement_date", "bmi", "source"])
    .join(cohort_keys.lazy(), on="person_id", how="inner")
    .with_columns(
        (pl.col("measurement_date") - pl.col("landmark_start_date"))
        .dt.total_days().alias("bmi_days_from_start")
    )
    .filter(pl.col("bmi_days_from_start").is_between(-730, 182))
    .with_columns(pl.col("bmi_days_from_start").abs().alias("days_diff_abs"))
    .group_by("person_id")
    .agg([
        pl.col("bmi").sort_by("days_diff_abs").first().alias("baseline_bmi"),
        pl.col("source").sort_by("days_diff_abs").first().alias("bmi_source"),
        pl.col("bmi_days_from_start").sort_by("days_diff_abs").first().alias("bmi_timing_days"),
    ])
)

cohort = (
    cohort.join(best_bmi, on="person_id", how="left")
    .with_columns([
        pl.col("baseline_bmi").is_not_null().cast(pl.Int8).fill_null(0).alias("has_bmi_flag"),
        pl.col("bmi_source").fill_null("Missing"),
        pl.col("bmi_timing_days").fill_null(0),
    ])
)


cohort, sjl_q1, sjl_q2 = add_sjl_tertiles(cohort)
consort["sjl_tertile_cutoffs"] = {"q1_hours": round(sjl_q1, 3), "q2_hours": round(sjl_q2, 3)}

cohort = cohort.with_columns([
    pl.col("followup_days").cast(pl.Int32),
    pl.col("censoring_time_years").cast(pl.Float32),
    pl.col("ehr_length_years").cast(pl.Float32),
    pl.col("avg_sjl_hours").cast(pl.Float32),
    pl.col("avg_sleep_duration_hours").cast(pl.Float32),
    pl.col("median_daily_steps").cast(pl.Int32),
    pl.col("baseline_bmi").cast(pl.Float32),
    pl.col("has_bmi_flag").cast(pl.Int8),
    pl.col("apnea_flag").cast(pl.Int8),
]).rename({
    "Income: Annual Income_detail":          "income_detail",
    "Education Level: Highest Grade_detail": "education_detail",
    "Employment: Employment Status_detail":  "employment_detail",
})

COHORT_FULL = f"{path_ck}/cohort_full.parquet"
cohort.sink_parquet(COHORT_FULL)
gc.collect()
cohort = pl.scan_parquet(COHORT_FULL)

def cast_cats(lf: pl.LazyFrame) -> pl.LazyFrame:
    """Cast all Categorical columns to Utf8 so R arrow::read_parquet works."""
    schema = lf.collect_schema()
    cat_cols = [c for c, dt in schema.items() if dt == pl.Categorical]
    if cat_cols:
        lf = lf.with_columns([pl.col(c).cast(pl.Utf8) for c in cat_cols])
    return lf


print("Step 13: Writing analytic cohort TSV and R input...")

schema = cohort.collect_schema()

SURVEY_COL_RENAMES = {
    "Race: What Race Ethnicity":          "race",
    "Smoking: 100 Cigs Lifetime":         "smoke",
    "Alcohol":                            "alcohol",
    "Education Level: Highest Grade":     "edu",
    "Income: Annual Income":              "income",
    "Employment: Employment Status":      "emp",
}

cohort_renamed = cohort.rename({
    k: v for k, v in SURVEY_COL_RENAMES.items()
    if k in cohort.collect_schema().names()
})

survey_cat_cols = [
    c for c, dt in cohort_renamed.collect_schema().items()
    if dt == pl.Categorical
    and c not in {"person_id", "landmark_start_date", "landmark_end_date"}
    and not c.endswith("_detail")
]

cohort_encoded_final = lazy_to_dummies(cohort_renamed, survey_cat_cols)

FINAL_TSV = f"{path_base}/invariant/cohort_with_landmark.tsv"
cohort_encoded_final.sink_csv(FINAL_TSV, separator="\t")

cast_cats(cohort_encoded_final).sink_parquet(f"{path_out}/fig_cohort_for_r.parquet")
gc.collect()

Phecode.add_phecode_time_to_event(
    phecode_count_file_path=phecode_file,
    cohort_file_path=FINAL_TSV,
    study_start_date_col="landmark_end_date",
    time_unit="years",
    output_file_path=f"{path_base}/invariant/phecode_counts_with_time.tsv",
)
print("Analytic cohort TSV + time-to-event + R parquet written.")


print("Step 14: Computing calendar coverage...")

lf_all_days = (
    lf_win
    .with_columns(
        pl.int_ranges(0, LANDMARK_DAYS).alias("day_offset")
    )
    .explode("day_offset")
    .with_columns([
        (pl.col("landmark_start_date") + pl.duration(days=pl.col("day_offset")))
        .alias("sleep_date"),
        pl.col("day_offset").cast(pl.Int32),
    ])
    .with_columns(
        pl.col("sleep_date").dt.weekday().alias("weekday_num")  # Mon=1 ... Sun=7
    )
    .select(["person_id", "sleep_date", "day_offset", "weekday_num"])
)

lf_has_sleep = (
    lf_sleep_daily
    .join(lf_win, on="person_id", how="inner")
    .filter(pl.col("sleep_date").is_between(
        pl.col("landmark_start_date"), pl.col("landmark_end_date"), closed="both"
    ))
    .select(["person_id", "sleep_date", "day_type"])
    .with_columns(pl.lit(1).cast(pl.Int8).alias("has_data"))
)

calendar_coverage = (
    lf_all_days
    .join(lf_has_sleep, on=["person_id", "sleep_date"], how="left")
    .with_columns(pl.col("has_data").fill_null(0))
    .group_by(["day_offset", "weekday_num"])
    .agg([
        pl.col("has_data").mean().alias("pct_coverage"),
        pl.col("has_data").sum().alias("n_with_data"),
        pl.len().alias("n_total"),
    ])
    .sort("day_offset")
)

print("Step 15: Evaluating lazy graphs and writing all outputs...")

print("  -> Materializing heavy execution graphs (this will take a moment)...")
lm_weekly_long         = lm_weekly_long.collect().lazy()
alltime_weekly_long    = alltime_weekly_long.collect().lazy()
alltime_person_summary = alltime_person_summary.collect().lazy()
lm_person_summary      = lm_person_summary.collect().lazy()
calendar_coverage      = calendar_coverage.collect().lazy()
print("  -> Graphs materialized. Writing parquets...")

available = set(cohort.collect_schema().names())

gcs_write_json(consort, f"{path_out}/consort_counts.json")

table1_cols = [
    "person_id", "sjl_tertile",
    "age_at_landmark", "sex_at_birth", "deprivation_index",
    "baseline_bmi", "avg_sjl_hours", "sd_sjl_hours",
    "avg_sleep_duration_hours", "median_daily_steps",
    "ehr_length_years", "censoring_time_years", "followup_days",
    "apnea_flag", "has_bmi_flag",
    "income_detail", "education_detail", "employment_detail",
    "Race: What Race Ethnicity",
    "Smoking: 100 Cigs Lifetime",
    "Alcohol",
]
(
    cast_cats(cohort.select([c for c in table1_cols if c in available]))
    .sink_parquet(f"{path_out}/table1_cohort.parquet")
)

(
    cast_cats(cohort.select([c for c in [
        "person_id", "sjl_tertile",
        "avg_sjl_hours", "sd_sjl_hours", "n_weeks",
        "avg_sleep_duration_hours",
        "mean_msw_min", "mean_msf_min",
        "mean_dur_weekday_min", "mean_dur_weekend_min",
        "median_daily_steps",
    ] if c in available]))
    .sink_parquet(f"{path_out}/table2_sleep_phenotype.parquet")
)

(
    cast_cats(cohort.select(["person_id", "avg_sjl_hours", "sjl_tertile"]))
    .sink_parquet(f"{path_out}/fig2a_sjl_distribution.parquet")
)

(
    cast_cats(cohort.select([c for c in [
        "person_id", "mean_msw_min", "mean_msf_min",
        "avg_sjl_hours", "sjl_tertile",
        "Race: What Race Ethnicity",
    ] if c in available]))
    .sink_parquet(f"{path_out}/fig2b_polar_midpoints.parquet")
)

(
    cast_cats(lm_weekly_long.join(
        cohort.select(["person_id", "sjl_tertile", "avg_sjl_hours"]),
        on="person_id", how="inner"
    ))
    .sink_parquet(f"{path_out}/fig2c_weekly_sjl_heatmap.parquet")
)

(
    cast_cats(
        cohort.select(["person_id", "avg_sjl_hours", "sjl_tertile"])
        .join(
            alltime_person_summary.select([
                "person_id", "alltime_avg_sjl_hours", "alltime_n_weeks"
            ]),
            on="person_id", how="inner"
        )
    )
    .sink_parquet(f"{path_out}/fig_landmark_vs_alltime_sjl.parquet")
)

calendar_coverage.sink_parquet(f"{path_out}/fig_calendar_coverage.parquet")

(
    cast_cats(cohort.select([c for c in [
        "person_id", "avg_sjl_hours", "sjl_tertile",
        "Race: What Race Ethnicity",
    ] if c in available]))
    .sink_parquet(f"{path_out}/fig6a_sjl_by_race.parquet")
)



cast_cats(lm_weekly_long).sink_parquet(f"{path_out}/supp_weekly_sjl_long.parquet")
cast_cats(alltime_weekly_long).sink_parquet(f"{path_out}/supp_alltime_weekly_sjl_long.parquet")
lm_person_summary.sink_parquet(f"{path_out}/supp_person_sleep_summary.parquet")

print("All outputs written.")

print("\n=== PIPELINE COMPLETE ===")